# CNN Scraper and Crawler 

The notebook should be equipped to run block by block.

## Imports

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from datetime import datetime
import json
import csv

# Setup Chrome options for headless mode
options = Options()
options.add_argument("--headless")  # Runs Chrome in headless mode.
options.add_argument('--no-sandbox')  # # Bypass OS security model
options.add_argument('--disable-gpu')  # applicable to windows os only
options.add_argument('start-maximized')  
options.add_argument('disable-infobars')
options.add_argument('--disable-extensions')



## Crawler 

This is the crawler for the CNN News website articles. Key aspects about the scrawler, the article range is unfortunately hardcoded. We scrape about 1700 articles from newest to oldest and only store links for articles that fall within out project's desired range of 10/7/2023 - 12/31/2023

Within that range, CNN produces 2 types of articles, namely - 
1. News Articles: These are articles published on a given date
2. Live-Blogs : These includes live-coverage or summaries of all the articles 
                written about the Israel-Palestine war for a given day so we 
                ignore them. 

We only **include News Articles** in our analysis and ignore rest. 

In [ ]:
# Initialize WebDriver and WebDriverWait
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 10)  # Create a WebDriverWait object

#Article parameters
start_point = 0  # Starting point for the articles
end_point = 1700 #this is hardcoded by looking at the article range 
step = 100  # Number of articles to fetch in each iteration
cnn_article_dict = {}  # Dictionary where string dates are keys and values are the article_url lists 


# Date range setup: These are the start dates and the end dates 
# of our articles 
input_date_format = "%b %d, %Y"
start_date = datetime.strptime("10/07/2023", "%m/%d/%Y")
end_date = datetime.strptime("12/31/2023", "%m/%d/%Y")



# Scraping loop
for start_from in range(start_point, end_point, step):

    #we update the original url to extract the necessary articles 
    #this is slightly easier and more time effecient than scraping on page
    article_url = f"https://www.cnn.com/search?q=Israel+Palestine+Hamas+Gaza+Conflict&from={start_from}&size={step}&sort=newest&types=article&section=&page=1"
    driver.get(article_url)

    try:
        # Scroll to the bottom of the page
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # Wait for the articles container to be fully loaded
        articles_container = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.container_list-images-with-description__cards-wrapper")))
        
        # Then proceed to find articles
        articles = articles_container.find_elements(By.CSS_SELECTOR, "div.container__item--type-media-image")
        for article in articles:
            try:
                #extract the article link 
                link_element = article.find_element(By.CSS_SELECTOR, "a.container__link--type-NewsArticle")
                article_link = link_element.get_attribute('href')

                # Proceed with the rest of your logic...
                date_element = article.find_element(By.CSS_SELECTOR, "div.container__date")
                article_date_str = date_element.text
                article_date = datetime.strptime(article_date_str, input_date_format)
                
                #prints the articles
                print(article_link, article_date_str, article_date)

                #if the article falls in our date range we add it to our dict
                if (article_date <= end_date) and (article_date >= start_date): 
                    if article_date_str in cnn_article_dict: 
                        cnn_article_dict[article_date_str].append(article_link)
                    else: 
                        cnn_article_dict[article_date_str] = [article_link]
            
            #live-blog articles have no NewsArticle element
            #we deliberately choose to ignore them because they are updates on 
            #current articles and we are limiting our analysis to originally 
            #published articles 
            except NoSuchElementException: 
                print("Ignoring live-blog article")
    except TimeoutException:
        print("Page took too long to load or articles container not found.")

    except Exception as e:
        print(f"Error occurred: {e}")

# Closing the WebDriver
driver.quit()
# Print or process the collected article dictionary


## Storing Scraped Articles in a json
We store current CNN articles to a json for safety

In [ ]:
#cnn_article_dict #uncomment to view the dictionary 
filename = "cnn_news_links.json"
with open(filename, 'w') as file:
    json.dump(cnn_article_dict, file, indent=4)

print(f"Dictionary saved to {filename}")


### CNN Scraper

The CNN scraper extracts the article title and only the necessary elements of 
the article body. We ignore author information and other unnecessary info 
that is not relevant to article body. 

In [ ]:
def cnn_scraper(article_url, article_date, article_index, driver, num_scraped):
    """
    Scrapes the cnn article title and body for a given article cnn article link. 

    INPUT: 
        article_url (str) : cnn article link 
        article_date (str): cnn article date, used in data outputting dict
        article_index (int): we keep track of this for debugging purposes if our 
                            code fails 
        driver (chromedriver obj): driver obj used to make calls and decisions 
        num_scraped (int): For logging purposes, keeps track of the number of 
                            articles successfully scraped, used in print statements 
    RETURN: 
        article_info_dict (dict): dict with necessary article info
    """
    
    print(f"Scraping article for date: {article_date}, URL: {article_url}, article_index = {article_index}")
    wait = WebDriverWait(driver, 10)  # Initialize WebDriverWait once and reuse it

    try:
        driver.get(article_url)

        # Scroll to the bottom to ensure all content is loaded
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait for the title of the article to be available
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "title")))
        article_title = driver.title.split("|")[0]
        print(f"Article title: {article_title}")

        # Wait for the paragraphs to be present
        wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "p.paragraph.inline-placeholder"))
        )

        # Now, it's safe to find the paragraphs as they are expected to be present
        paragraphs = driver.find_elements(By.CSS_SELECTOR, "p.paragraph.inline-placeholder")

        # Loop through found <p> elements and extract text
        article_paragraphs = [paragraph.text for paragraph in paragraphs]
        article_text = " ".join(article_paragraphs) #join all paragraphs
        
        #output them as an info dictionary 
        article_info_dict = {
            "article_url": article_url,
            "article_date": article_date,
            "article_title": article_title,
            "all_text": article_text
        }

        print(f"Successfully retrieved article content: num_scraped = {num_scraped}")
        return article_info_dict
    
    except TimeoutException:
        print(f"Timeout occurred while loading URL: {article_url}, date = {article_date}, index = {article_index}")
        return None
    except Exception as e:
        print(f"An error occurred for URL: {article_url} - {str(e)}, date = {article_date}, index = {article_index}")
        return None
    



## Running the Scraper 

The iteratively calls the scraper for all article links and concurrently stores the information in a csv file. 

### Load the Stored Links (optional)
Run the code below if need be, not necessary if you already ran the code blocks above

In [ ]:
filename = 'cnn_news_links.json'

# Open the file and load its content into a Python dictionary
with open(filename, 'r') as file:
    cnn_articles_dict = json.load(file)


### Iteratively running the scraper

We restart the chromedriver instance for every 10 articles scraped. We also keep track of articles that we were unable to successfully scrape. This is incase we want to retry scraping them 

In [ ]:
#We keep track of urls we were unable to scrape
error_urls = []


count = 0 #tracks the number of articles scaped 
restart_driver_count = 10 #restart the driver every 10 urls scraped
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

#uncomment both error_point and error_count if code breaks 
#set the error point to the last count index 
#to the error 
#error_point = 130 
#error_count = 0 


# Initialize the CSV file for writing
csv_filename = 'scraped_cnn_news_articles.csv'

with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    
    # The fieldnames will be updated inside the loop
    writer = csv.DictWriter(csvfile, fieldnames=[], quoting=csv.QUOTE_ALL)  # Ensure all fields are quoted
    writer.writeheader()  # Placeholder header, will be updated

    #for each date in our cnn_articles dict 
    for date, article_urls in cnn_articles_dict.items():

        #we keep track of each article_url and index incase our scraper crashes 
        #we can get better data on what was the last article scraped 
        #also we keep track of counts so we can run our scraper from the last 
        #checkpoint rather than restarting it. 
        for index, article_url in enumerate(article_urls):
            
            #uncomment the code below if you run into issues
            # error_count+=1 
            # if error_count <= error_point: 
            #     continue

            #We restart the driver every 10 links scraped 
            if count % restart_driver_count == 0 and count != 0: 
                driver.quit()
                driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

            #Get the article info
            article_info = cnn_scraper(article_url, article_date, index, driver, count)

            #if the dict exists we update the csv 
            if article_info:

                # Update the writer with the correct fieldnames based on the first article
                if count == 0:
                    writer.fieldnames = article_info.keys()
                    writer.writeheader()  # Write the correct header based on the first article info
                
                #write the row for the article
                writer.writerow(article_info)
            
            #if we had an error we update the error urls
            else:
                error_urls.append(article_url)      
            count += 1

#quit the driver instance at the very end
driver.quit()